En este cuaderno vamos a usar un modelo ViT (Vision Transformers). El elegido es: *google/vit-base-patch16-224-in21k*

In [2]:
!pip install evaluate decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 149.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from sklearn.model_selection import GroupShuffleSplit
from datasets import Dataset, Image, Features, Value
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import evaluate

# 1. Fase de Entrenamiento Train/Valid/Test

## 1.1 Defiendo las rutas y el modelo con el que vamos a trabajar

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
CSV_IMAGENES_TRAIN = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/dataset_imagenes_train.csv"
CSV_TRAIN_MASTER_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1_master.csv"
CSV_TEST_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"
CSV_IMAGENES_TEST = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/dataset_imagenes_test.csv"

OUTPUT_DIR = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/ViT_FineTuned"
MODEL_CHECKPOINT = "google/vit-base-patch16-224-in21k"

In [ ]:
print("Cargando el dataset de imágenes y las plantillas de texto...")
df_imagenes = pd.read_csv(CSV_IMAGENES)
df_train_text = pd.read_csv(CSV_TRAIN_MASTER_TEXT)
df_test_text = pd.read_csv(CSV_TEST_TEXT)

Cargando el dataset de imágenes y las plantillas de texto...


## 1.2 Particionado

**OJO** con este paso, tendremos que tener en cuenta el contenido de la columna *id_EXIST* porque estaríamos falseando los resultados si de un mismo vídeo tenemos un frame en el conjunto de entrenamiento, otro frame en el conjunto de test y otro frame en el conjunto de validación.

In [ ]:
# =================================================================
# FASE 1: Alineación 80/20 Estática Multimodal
# =================================================================
# Extraemos los IDs de los vídeos para que el particionado de imágenes
# sea EXACTAMENTE idéntico al de texto.
train_master_ids = df_train_text['id_EXIST'].unique()
test_ids = df_test_text['id_EXIST'].unique()

# Filtramos las imágenes basándonos en las particiones estáticas
df_train_master = df_imagenes[df_imagenes['id_EXIST'].isin(train_master_ids)].copy()
test_df = df_imagenes[df_imagenes['id_EXIST'].isin(test_ids)].copy()

# =================================================================
# FASE 2: División Dinámica del Train Master (90% Train / 10% Valid)
# =================================================================
# Usamos GroupShuffleSplit para asegurar que todos los fotogramas
# de un mismo vídeo caen en el MISMO bloque (evita Data Leakage)
gss_train_val = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)

# Separamos Train (90%) y Valid (10%)
train_idx, val_idx = next(gss_train_val.split(df_train_master, groups=df_train_master['id_EXIST']))

train_df = df_train_master.iloc[train_idx].copy()
val_df = df_train_master.iloc[val_idx].copy()

print("\n--- DISTRIBUCIÓN DEL DATASET DE IMÁGENES ---")
print(f"Vídeos en Train: {train_df['id_EXIST'].nunique()} (Aprox {len(train_df)} fotogramas)")
print(train_df['label'].value_counts())

print(f"\nVídeos en Valid: {val_df['id_EXIST'].nunique()} (Aprox {len(val_df)} fotogramas)")
print(val_df['label'].value_counts())

print(f"\nVídeos en Test (Intocable): {test_df['id_EXIST'].nunique()} (Aprox {len(test_df)} fotogramas)")
print(test_df['label'].value_counts())


--- DISTRIBUCIÓN DEL DATASET DE IMÁGENES ---
Vídeos en Train: 1805 (Aprox 7218 fotogramas)
label
0    3750
1    3468
Name: count, dtype: int64

Vídeos en Valid: 201 (Aprox 804 fotogramas)
label
0    428
1    376
Name: count, dtype: int64

Vídeos en Test (Intocable): 502 (Aprox 2008 fotogramas)
label
0    1044
1     964
Name: count, dtype: int64


## 1.3 Conversión a Formato `Dataset` de Hugging Face

In [ ]:
def crear_dataset(dataframe):
    # Primero creamos el dataset leyendo las rutas como texto plano desde el dataframe
    dataset = Dataset.from_pandas(dataframe)

    # Casteamos la columna de texto a tipo Image() para que lea los archivos de Drive
    dataset = dataset.cast_column("path_imagen", Image())

    # Renombramos para que el modelo lo entienda
    return dataset.rename_column("path_imagen", "image")

print("Transformando rutas en imágenes procesables...")
train_dataset = crear_dataset(train_df)
valid_dataset = crear_dataset(val_df)

Transformando rutas en imágenes procesables...


## 1.4 Procesador de Imágenes (`ViTImageProcessor`)

Esto es igual que el tokenizador de texto, pero para imágenes (redimensiona a 224x224, normaliza colores, etc.)

In [ ]:
processor = ViTImageProcessor.from_pretrained(MODEL_CHECKPOINT)

def process_images(batch):
    # Toma una lista de imágenes y las convierte en los tensores matemáticos que ViT necesita
    inputs = processor([img.convert("RGB") for img in batch["image"]], return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs

# Aplicamos la transformación "al vuelo" para no saturar la RAM
train_dataset.set_transform(process_images)
valid_dataset.set_transform(process_images)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

## 1.5 Inicialización del Modelo Base ViT

In [ ]:
id2label = {0: "No misógino", 1: "Misógino"}
label2id = {"No misógino": 0, "Misógino": 1}

model = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True # Necesario porque el modelo in21k original no tiene capa de clasificación
)

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.intermediate.dense.weight        | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.bias     | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.bias   | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.bias          | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.weight           | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.bias                | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.weight              | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.weight          | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.bias   | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.

## 1.6 Métrica de Evaluación

In [ ]:
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

# Usamos tu amado F1 Score macro
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}

## 1.7 Hiperparámetros del Entrenamiento

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False, # IMPORTANTE: En visión debe ser False
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,          # ViT prefiere Learning Rates más bajos que los LLMs
    per_device_train_batch_size=16, # ViT es ligero, aguanta un batch de 16 en Colab
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    fp16=True,                   # Precisión mixta para acelerar en GPU
    report_to="none"
)

## 1.8 Entrenamiento

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("🚀 Iniciando entrenamiento visual con ViT...")
trainer.train()

print("💾 Guardando modelo visual...")
trainer.save_model(OUTPUT_DIR + "/modelo_final")
processor.save_pretrained(OUTPUT_DIR + "/modelo_final")
print("✅ ¡Entrenamiento completado!")

🚀 Iniciando entrenamiento visual con ViT...


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.698574,0.688308,0.347403,0.532338
2,0.691143,0.689215,0.352714,0.533582
3,0.687662,0.685212,0.456870,0.547264
4,0.688888,0.684709,0.514558,0.552239
5,0.680045,0.681226,0.519046,0.565920


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Guardando modelo visual...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ ¡Entrenamiento completado!


# 2. Fase de Inferencia/Evaluación

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. Configuración y carga de datos de TEST

CSV_IMAGENES_TEST = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/dataset_imagenes_test.csv"
MODEL_DIR = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/ViT_FineTuned/modelo_final"

# ARCHIVOS DE SALIDA SEPARADOS
DIR_SALIDA = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/"
CSV_SALIDA_MAX = os.path.join(DIR_SALIDA, "predicciones_vit_test_max.csv")
CSV_SALIDA_MEAN = os.path.join(DIR_SALIDA, "predicciones_vit_test_mean.csv")
CSV_SALIDA_MAJORITY = os.path.join(DIR_SALIDA, "predicciones_vit_test_majority.csv")

print("Cargando el dataset estático de test...")
test_df = pd.read_csv(CSV_IMAGENES_TEST)

test_df

Cargando el dataset estático de test...


,id_EXIST,path_imagen,label
0,120783,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,0
1,120783,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,0
2,120783,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,0
3,120783,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,0
4,220502,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,1
...,...,...,...
2003,121242,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,1
2004,121515,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,1
2005,121515,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,1
2006,121515,/content/drive/MyDrive/dataset/EXIST 2025 Vide...,1


In [ ]:
# 2. Cargamos el modelo que tenemos entrenado

print("Cargando procesador y modelo ViT entrenado...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = ViTImageProcessor.from_pretrained(MODEL_DIR)
model = ViTForImageClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

print(f"Iniciando inferencia sobre {test_df['id_EXIST'].nunique()} vídeos de test...")

Cargando procesador y modelo ViT entrenado...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Iniciando inferencia sobre 502 vídeos de test...


In [ ]:
predicciones_por_video = {}

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    id_vid = row['id_EXIST']
    img_path = row['path_imagen']
    true_label = row['label']

    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"⚠️ Error cargando imagen {img_path}: {e}")
        continue

    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
        prob_misogino = probs[1].item()

    if id_vid not in predicciones_por_video:
        predicciones_por_video[id_vid] = {'probs': [], 'true_label': true_label}

    predicciones_por_video[id_vid]['probs'].append(prob_misogino)

100%|██████████| 2008/2008 [26:44<00:00,  1.25it/s]


## 2.1 Max-Pooling

Con que solamente 1 solo fotograma de los 4 sea etiquetado como misógino, el video entero se marcará como misógino.

In [ ]:
y_true = []
y_pred_max = []

# Listas independientes para cada CSV
resultados_max = []


for id_vid, data in predicciones_por_video.items():
    true_label = data['true_label']
    y_true.append(true_label)

    probabilidades = data['probs']
    predicciones_binarias = [1 if p > 0.5 else 0 for p in probabilidades]

    # --- 1. Max-Pooling ---
    max_prob = max(probabilidades) if probabilidades else 0.0
    pred_max = 1 if max_prob > 0.5 else 0
    y_pred_max.append(pred_max)

    resultados_max.append({
        "id_EXIST": id_vid, "prob_misogino": max_prob,
        "prediccion_binaria": pred_max, "label_real": true_label
    })

In [ ]:
os.makedirs(DIR_SALIDA, exist_ok=True)
pd.DataFrame(resultados_max).to_csv(CSV_SALIDA_MAX, index=False)

print("\n✅ ¡Archivo CSV Max-Pooling guardados correctamente!")
print(f" - {CSV_SALIDA_MAX}")


✅ ¡Archivo CSV Max-Pooling guardados correctamente!
 - /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/predicciones_vit_test_max.csv


In [ ]:
print("\n" + "="*50)
print("🏆 RESULTADOS TEST ESTÁTICO (MAX-POOLING)")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred_max, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred_max):.4f}")


🏆 RESULTADOS TEST ESTÁTICO (MAX-POOLING)
F1-Score (Macro): 0.4997
Accuracy: 0.5199


In [ ]:
print("\nMatriz de Confusión (Max-Pooling):\n", confusion_matrix(y_true, y_pred_max))
print("\nClassification Report (Max-Pooling):\n", classification_report(y_true, y_pred_max, target_names=["No Misógino", "Misógino"]))


Matriz de Confusión (Max-Pooling):
 [[181  80]
 [161  80]]

Classification Report (Max-Pooling):
               precision    recall  f1-score   support

 No Misógino       0.53      0.69      0.60       261
    Misógino       0.50      0.33      0.40       241

    accuracy                           0.52       502
   macro avg       0.51      0.51      0.50       502
weighted avg       0.52      0.52      0.50       502



## 2.2 Average Pooling

Se cogerá el porcentaje de seguridad del modelo para los 4 fotogramas y haremos la media. Si la media >= 50% se cataloga vídeo como misógino.

In [ ]:
y_true = []
y_pred_mean = []

# Listas independientes para cada CSV
resultados_mean = []

for id_vid, data in predicciones_por_video.items():
    true_label = data['true_label']
    y_true.append(true_label)

    probabilidades = data['probs']
    predicciones_binarias = [1 if p > 0.5 else 0 for p in probabilidades]

    # --- 2. Mean-Pooling ---
    mean_prob = sum(probabilidades) / len(probabilidades) if probabilidades else 0.0
    pred_mean = 1 if mean_prob > 0.5 else 0
    y_pred_mean.append(pred_mean)

    resultados_mean.append({
        "id_EXIST": id_vid, "prob_misogino": mean_prob,
        "prediccion_binaria": pred_mean, "label_real": true_label
    })

In [ ]:
os.makedirs(DIR_SALIDA, exist_ok=True)
pd.DataFrame(resultados_mean).to_csv(CSV_SALIDA_MEAN, index=False)

print("\n✅ ¡Archivo CSV Mean-pooling guardado correctamente!")
print(f" - {CSV_SALIDA_MEAN}")



✅ ¡Archivo CSV Mean-pooling guardado correctamente!
 - /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/predicciones_vit_test_mean.csv


In [ ]:
print("\n" + "="*50)
print("📊 RESULTADOS TEST ESTÁTICO (MEAN-POOLING)")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred_mean, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred_mean):.4f}")


📊 RESULTADOS TEST ESTÁTICO (MEAN-POOLING)
F1-Score (Macro): 0.4915
Accuracy: 0.5438


In [ ]:
print("\nMatriz de Confusión (Mean-Pooling):\n", confusion_matrix(y_true, y_pred_mean))
print("\nClassification Report (Mean-Pooling):\n", classification_report(y_true, y_pred_mean, target_names=["No Misógino", "Misógino"]))


Matriz de Confusión (Mean-Pooling):
 [[217  44]
 [185  56]]

Classification Report (Mean-Pooling):
               precision    recall  f1-score   support

 No Misógino       0.54      0.83      0.65       261
    Misógino       0.56      0.23      0.33       241

    accuracy                           0.54       502
   macro avg       0.55      0.53      0.49       502
weighted avg       0.55      0.54      0.50       502



## 2.3 Votación por mayoría

Hacemos que al menos 2 o 3 fotogramas sean clasificados como misóginos para considerar el vídeo como misógino.

In [ ]:
y_true = []
y_pred_majority = []

# Listas independientes para cada CSV
resultados_majority = []

for id_vid, data in predicciones_por_video.items():
    true_label = data['true_label']
    y_true.append(true_label)

    probabilidades = data['probs']
    predicciones_binarias = [1 if p > 0.5 else 0 for p in probabilidades]

    # --- 3. Votación por Mayoría ---
    votos_misogino = sum(predicciones_binarias)
    total_frames = len(predicciones_binarias)
    prob_majority = votos_misogino / total_frames if total_frames > 0 else 0.0

    mitad_frames = total_frames / 2
    if votos_misogino >= mitad_frames:
        pred_majority = 1
    else:
        pred_majority = 0
    y_pred_majority.append(pred_majority)

    resultados_majority.append({
        "id_EXIST": id_vid, "prob_misogino": prob_majority,
        "prediccion_binaria": pred_majority, "label_real": true_label
    })

In [ ]:
os.makedirs(DIR_SALIDA, exist_ok=True)
pd.DataFrame(resultados_majority).to_csv(CSV_SALIDA_MAJORITY, index=False)

print("\n✅ ¡Archivo CSV Votacion por mayoria guardado correctamente!")
print(f" - {CSV_SALIDA_MAJORITY}")


✅ ¡Archivo CSV Votacion por mayoria guardado correctamente!
 - /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/predicciones_vit_test_majority.csv


In [ ]:
print("\n" + "="*50)
print("🗳️ RESULTADOS TEST ESTÁTICO (MAJORITY VOTE)")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred_majority, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred_majority):.4f}")


🗳️ RESULTADOS TEST ESTÁTICO (MAJORITY VOTE)
F1-Score (Macro): 0.4760
Accuracy: 0.5179


In [ ]:
print("\nMatriz de Confusión (Majority-Vote):\n", confusion_matrix(y_true, y_pred_majority))
print("\nClassification Report (Majority-Vote):\n", classification_report(y_true, y_pred_majority, target_names=["No Misógino", "Misógino"]))


Matriz de Confusión (Majority-Vote):
 [[201  60]
 [182  59]]

Classification Report (Majority-Vote):
               precision    recall  f1-score   support

 No Misógino       0.52      0.77      0.62       261
    Misógino       0.50      0.24      0.33       241

    accuracy                           0.52       502
   macro avg       0.51      0.51      0.48       502
weighted avg       0.51      0.52      0.48       502



# 3. Entrenamiento con el 100% de los datos

## 3.1 Configuración y rutas

In [ ]:
MODEL_CHECKPOINT = "google/vit-base-patch16-224-in21k"

# CSV original con TODAS las imágenes (100% de los datos de entrenamiento)
CSV_TODAS_IMAGENES = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/dataset_imagenes_all.csv"

# Carpeta de salida para el modelo definitivo
OUTPUT_DIR = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/SubidasCompeticion/ViT_Final_100"

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 3.2. Carga y preparación del 100% del dataset

In [ ]:
print("Cargando el 100% de las imágenes de entrenamiento...")
df_completo = pd.read_csv(CSV_TODAS_IMAGENES)

Cargando el 100% de las imágenes de entrenamiento...


In [ ]:
# Verificamos distribución
print(f"Total de fotogramas a procesar: {len(df_completo)}")
print(df_completo['label'].value_counts())

Total de fotogramas a procesar: 10030
label
0    5222
1    4808
Name: count, dtype: int64


In [ ]:
# Función helper para transformar el dataframe en el Dataset de HuggingFace
def crear_dataset(dataframe):
    dataset = Dataset.from_pandas(dataframe)
    # Casting vital para que HuggingFace entienda que 'path_imagen' apunta a un archivo físico
    dataset = dataset.cast_column("path_imagen", Image())
    return dataset.rename_column("path_imagen", "image")

train_dataset = crear_dataset(df_completo)

## 3.3. Procesador de imágenes

In [ ]:
print(f"Cargando el procesador de imágenes para {MODEL_CHECKPOINT}...")
processor = ViTImageProcessor.from_pretrained(MODEL_CHECKPOINT)

Cargando el procesador de imágenes para google/vit-base-patch16-224-in21k...


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [ ]:
def process_images(batch):
    # Toma una lista de imágenes (ya cargadas por el Dataset) y las convierte a tensores de píxeles
    inputs = processor([img.convert("RGB") for img in batch["image"]], return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs

# Usamos set_transform para que el procesamiento se haga "al vuelo" y no sature la RAM
train_dataset.set_transform(process_images)

##3.4 Inicialización del modelo ViT

In [ ]:
print("Inicializando la arquitectura ViT y la nueva cabeza de clasificación...")
id2label = {0: "No misógino", 1: "Misógino"}
label2id = {"No misógino": 0, "Misógino": 1}

Inicializando la arquitectura ViT y la nueva cabeza de clasificación...


In [ ]:
model = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True # VITAL: Obliga a desechar cabezas anteriores y crear una de 2 salidas
)

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.attention.attention.key.weight   | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.bias   | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.weight              | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.bias   | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.bias                | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.bias     | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
encoder.layer.{0...11}.attention.output.dense.weight    | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.bias          | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.weight          | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.bias

In [ ]:
# Collate function para empaquetar los tensores generados en el paso 3
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

## 3.5 Entrenamiento a ciegas (con el 100% de los datos)

In [ ]:
# Mantenemos las 5 épocas que tenías en tu código original, ya que ViT suele ser más resistente al sobreajuste que los LLMs
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False, # ¡CRUCIAL EN VISIÓN! Si es True, borra las imágenes antes de procesarlas
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,                   # Aceleración en L4/T4
    logging_strategy="epoch",

    # Apagamos evaluación y guardado intermedio
    eval_strategy="no",
    save_strategy="no",
    report_to="none",

    # Optimizaciones de disco
    dataloader_num_workers=2,
    dataloader_pin_memory=True
)

In [ ]:
print(train_dataset)

Dataset({
    features: ['id_EXIST', 'image', 'label'],
    num_rows: 10030
})


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
    # Ya no hay valid_dataset ni compute_metrics
)

print("🚀 Lanzando entrenamiento visual definitivo...")
trainer.train()

print(f"💾 Guardando el modelo y procesador final en: {OUTPUT_DIR}")
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print("✅ ¡Entrenamiento completado!")

🚀 Lanzando entrenamiento visual definitivo...


Step,Training Loss
627,0.696169
1254,0.693853
1881,0.691543
2508,0.688044
3135,0.681271


💾 Guardando el modelo y procesador final en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/SubidasCompeticion/ViT_Final_100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ ¡Entrenamiento completado!


# 4. Script de predicción contra fichero test competición y generación JSON

In [3]:
import pandas as pd
import numpy as np
import torch
import json
import os
from tqdm import tqdm
from decord import VideoReader, cpu
import decord
from transformers import ViTImageProcessor, ViTForImageClassification

In [4]:
# Silenciamos logs de decord y configuramos el puente a pytorch
decord.bridge.set_bridge('torch')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 4.1. Configuración y rutas

In [5]:
# Ruta a tu ViT definitivo entrenado al 100%
RUTA_MODELO_FINAL = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/SubidasCompeticion/ViT_Final_100"

# Ruta al Test Oficial Limpio (formato CSV)
RUTA_TEST_JSON = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/test/EXIST2025_test_clean.json"
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/test/"

RUTA_SUBMISSION_JSON = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/SubidasCompeticion/vit_test_oficial.json"

## 4.2. Carga del modelos y de los datos

In [6]:
print("Cargando el test oficial limpio...")
df_test = pd.read_json(RUTA_TEST_JSON, orient='index')
if "Unnamed: 0" in df_test.columns:
    df_test.drop(columns=["Unnamed: 0"], inplace=True)

print(f"Cargando Procesador y Modelo ViT desde {RUTA_MODELO_FINAL}...")
processor = ViTImageProcessor.from_pretrained(RUTA_MODELO_FINAL)
model = ViTForImageClassification.from_pretrained(RUTA_MODELO_FINAL).to(device)
model.eval()

NUM_FRAMES = 4 # El número de fotogramas que evaluaremos por vídeo

Cargando el test oficial limpio...
Cargando Procesador y Modelo ViT desde /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/SubidasCompeticion/ViT_Final_100...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

## 4.3. Inferencia "al vuelo" con max-pooling batcheado

In [8]:
print(f"Realizando predicciones sobre {len(df_test)} vídeos...")
predicciones = []

for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    id_vid = row["id_EXIST"]
    val_str = str(row.get('path_video', id_vid))

    # Construcción robusta de la ruta
    if not val_str.endswith(".mp4"):
        val_str += ".mp4"
    if "videos/" in val_str:
        ruta_video = os.path.join(RUTA_BASE_VIDEOS, val_str)
    else:
        ruta_video = os.path.join(RUTA_BASE_VIDEOS, "videos", val_str)

    prob_misogino_video = 0.5 # Valor neutro de seguridad si el vídeo está corrupto

    try:
        # Abrimos el vídeo en memoria RAM
        vr = VideoReader(ruta_video, ctx=cpu(0))
        total_frames = len(vr)

        # Calculamos 4 índices equidistantes
        if total_frames > NUM_FRAMES:
            step = total_frames // (NUM_FRAMES + 1)
            frame_indices = [step * i for i in range(1, NUM_FRAMES + 1)]
        else:
            # Si el video tiene menos de 4 frames, cogemos los que haya
            frame_indices = list(range(total_frames))

        # Extraemos los fotogramas y los pasamos a numpy array
        frames = vr.get_batch(frame_indices).numpy()

        # Le pasamos la lista de frames al procesador de golpe (Batch processing)
        inputs = processor(list(frames), return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            # Calculamos las probabilidades [N_frames, 2]
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            # Extraemos la columna de la clase "Misógino" (índice 1) para todos los frames
            probs_misogino_frames = probs[:, 1].tolist()

        # MAX-POOLING: Nos quedamos con la probabilidad del frame más alto
        if probs_misogino_frames:
            prob_misogino_video = max(probs_misogino_frames)

    except Exception as e:
        # Ignoramos el error en consola para no ensuciar la salida, asignará 0.5
        pass

    pred_binaria = 1 if prob_misogino_video > 0.5 else 0
    predicciones.append(pred_binaria)

df_test["predicted_label"] = predicciones

Realizando predicciones sobre 674 vídeos...


100%|██████████| 674/674 [10:19<00:00,  1.09it/s]


## 4.4. Formateo al estándar PyEvall (JSON final)

In [9]:
print("\nGenerando archivo de sumisión en formato PyEvALL...")
output_json = []
for idx, row in df_test.iterrows():
    entry = {
        "test_case": "EXIST2025",
        "id": str(row["id_EXIST"]),
        "value": "YES" if int(row["predicted_label"]) == 1 else "NO"
    }
    output_json.append(entry)

os.makedirs(os.path.dirname(RUTA_SUBMISSION_JSON), exist_ok=True)
with open(RUTA_SUBMISSION_JSON, "w", encoding="utf-8") as f:
    json.dump(output_json, f, indent=2)

print(f"✅ ¡Completado! Archivo guardado en: {RUTA_SUBMISSION_JSON}")


Generando archivo de sumisión en formato PyEvALL...
✅ ¡Completado! Archivo guardado en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/SubidasCompeticion/vit_test_oficial.json


In [11]:
df_test['predicted_label'].value_counts()

,count
predicted_label,
0,382
1,292
